# Solar Valuation - Data Exploration

This notebook demonstrates the data engineering pipeline:
1. Fetching live electricity prices from UK National Grid ESO
2. Fetching solar irradiance data from Open-Meteo
3. Fetching carbon prices (EU ETS) for emissions cost modeling
4. Storing and querying with DuckDB
5. Basic analysis patterns

In [ ]:
import sys
sys.path.insert(0, "..")

import asyncio
from datetime import datetime, timedelta

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Fetch Live Electricity Prices

Using the NGESO (National Grid ESO) API to get GB imbalance prices.

In [ ]:
from src.data.ngeso_client import NGESOClient

async def fetch_prices():
    """Fetch last 7 days of GB imbalance prices."""
    start = datetime.utcnow() - timedelta(days=7)
    
    async with NGESOClient() as client:
        prices = await client.get_imbalance_prices(start)
    
    return prices

prices = await fetch_prices()
print(f"Fetched {len(prices)} price records")

# Convert to DataFrame for analysis
prices_df = pl.DataFrame([p.model_dump() for p in prices])
prices_df.head()

In [ ]:
# Plot price time series
fig = px.line(
    prices_df.to_pandas(),
    x="timestamp",
    y="price_mwh",
    title="GB Electricity Imbalance Prices (£/MWh)",
)
fig.update_layout(xaxis_title="Time", yaxis_title="Price (£/MWh)")
fig.show()

## 2. Fetch Solar Irradiance Data

Using Open-Meteo API for historical solar radiation at a UK site.

In [ ]:
from src.data.openmeteo_client import OpenMeteoClient

# Coordinates for a hypothetical solar farm in Oxfordshire
SITE_LAT = 51.75
SITE_LON = -1.25

async def fetch_irradiance():
    """Fetch last 30 days of solar irradiance."""
    end = datetime.utcnow()
    start = end - timedelta(days=30)
    
    async with OpenMeteoClient() as client:
        records = await client.get_historical_solar(
            SITE_LAT, SITE_LON, start, end
        )
    
    return records

irradiance = await fetch_irradiance()
print(f"Fetched {len(irradiance)} irradiance records")

irradiance_df = pl.DataFrame([r.model_dump() for r in irradiance])
irradiance_df.head()

In [ ]:
# Plot irradiance with temperature
fig = make_subplots(specs=[[{"secondary_y": True}]])

pdf = irradiance_df.to_pandas()

fig.add_trace(
    go.Scatter(x=pdf["timestamp"], y=pdf["ghi_wm2"], name="GHI (W/m²)"),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=pdf["timestamp"], y=pdf["temperature_c"], name="Temperature (°C)"),
    secondary_y=True,
)

fig.update_layout(title="Solar Irradiance & Temperature")
fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="GHI (W/m²)", secondary_y=False)
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True)
fig.show()

## 3. Fetch Carbon Prices (EU ETS)

Carbon prices are critical for solar project valuation:
- They represent the **avoided cost** of CO2 emissions
- Solar generation displaces fossil fuel generation
- Higher carbon prices = higher value for renewable projects

The EU ETS (Emissions Trading System) is the world's largest carbon market (~€65-80/tCO2 in 2024).

In [ ]:
from src.data.carbon_client import get_fallback_carbon_prices, OilPriceAPIClient
from src.data.schemas import CarbonScheme
import os

# Try to fetch from API if key is available, otherwise use fallback data
api_key = os.environ.get("OILPRICE_API_KEY")

if api_key:
    async def fetch_carbon():
        async with OilPriceAPIClient(api_key) as client:
            return await client.get_latest_price(CarbonScheme.EU_ETS)
    
    latest_carbon = await fetch_carbon()
    print(f"Latest EU ETS price: €{latest_carbon.price_tonne:.2f}/tCO2")
else:
    print("No API key found, using historical fallback data")

# Load historical carbon prices (works without API key)
carbon_prices = get_fallback_carbon_prices(CarbonScheme.EU_ETS)
carbon_df = pl.DataFrame([c.model_dump() for c in carbon_prices])

print(f"\nLoaded {len(carbon_df)} monthly EU ETS price records")
carbon_df

In [ ]:
# Plot carbon price trend
fig = px.line(
    carbon_df.to_pandas(),
    x="timestamp",
    y="price_tonne",
    title="EU ETS Carbon Price (€/tCO2)",
    markers=True,
)
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Price (€/tCO2)",
    yaxis_range=[0, 100],
)
fig.add_hline(y=carbon_df["price_tonne"].mean(), line_dash="dash", 
              annotation_text=f"Mean: €{carbon_df['price_tonne'].mean():.1f}")
fig.show()

## 4. Store Data with DuckDB

Persist fetched data locally for fast analytical queries.

In [ ]:
from src.data.storage import PriceStore

store = PriceStore(db_path="../data/prices.duckdb")

# Insert all fetched data
prices_inserted = store.insert_prices(prices)
irradiance_inserted = store.insert_irradiance(irradiance)
carbon_inserted = store.insert_carbon(carbon_prices)

print(f"Inserted {prices_inserted} electricity price records")
print(f"Inserted {irradiance_inserted} irradiance records")
print(f"Inserted {carbon_inserted} carbon price records")

# Check stats
store.get_stats()

## 5. Query Patterns

Common analytical queries for valuation modeling.

In [ ]:
# Query stored prices
start = datetime.utcnow() - timedelta(days=7)
end = datetime.utcnow()

queried_prices = store.get_prices(start, end, source="ngeso")
print(f"Queried {len(queried_prices)} price records")

# Calculate daily statistics
daily_stats = queried_prices.with_columns(
    pl.col("timestamp").dt.date().alias("date")
).group_by("date").agg(
    pl.col("price_mwh").mean().alias("mean_price"),
    pl.col("price_mwh").std().alias("std_price"),
    pl.col("price_mwh").min().alias("min_price"),
    pl.col("price_mwh").max().alias("max_price"),
)

daily_stats

In [ ]:
# Calculate capacity factor from irradiance
# Typical mono-Si panel: ~200 W/m² at 1000 W/m² irradiance = 20% efficiency
PANEL_EFFICIENCY = 0.20
STC_IRRADIANCE = 1000  # W/m² at standard test conditions

irradiance_data = store.get_irradiance(start, end)

# Hourly capacity factor estimate
capacity_factors = irradiance_data.with_columns(
    (pl.col("ghi_wm2") / STC_IRRADIANCE * PANEL_EFFICIENCY).alias("capacity_factor"),
    pl.col("timestamp").dt.date().alias("date"),
)

daily_cf = capacity_factors.group_by("date").agg(
    pl.col("capacity_factor").mean().alias("daily_cf"),
    pl.col("ghi_wm2").sum().alias("daily_ghi_wh_m2"),  # Wh/m² per day
)

daily_cf

In [ ]:
# Get latest carbon price for valuation
latest_carbon_price = store.get_latest_carbon(scheme="EU_ETS")
print(f"Latest EU ETS carbon price: €{latest_carbon_price:.2f}/tCO2")

# Carbon value of solar generation
# UK grid carbon intensity ~200 gCO2/kWh (varies by time)
GRID_CARBON_INTENSITY = 0.2  # tCO2/MWh
SOLAR_CAPACITY_MW = 50  # Our hypothetical solar farm
AVG_CAPACITY_FACTOR = daily_cf["daily_cf"].mean()

# Annual generation estimate
annual_generation_mwh = SOLAR_CAPACITY_MW * AVG_CAPACITY_FACTOR * 8760
annual_emissions_avoided = annual_generation_mwh * GRID_CARBON_INTENSITY
annual_carbon_value = annual_emissions_avoided * latest_carbon_price

print(f"\n50MW Solar Farm Carbon Value Estimate:")
print(f"  Avg capacity factor: {AVG_CAPACITY_FACTOR:.1%}")
print(f"  Annual generation: {annual_generation_mwh:,.0f} MWh")
print(f"  Emissions avoided: {annual_emissions_avoided:,.0f} tCO2")
print(f"  Carbon value: €{annual_carbon_value:,.0f}/year")

## Next Steps

1. **Run backfill**: `python -m src.data.pipeline --backfill` to populate historical data
2. **Start scheduler**: `python -m src.data.scheduler` for continuous updates
3. **Get API keys** (optional):
   - [OilPriceAPI](https://www.oilpriceapi.com/auth/signup) - Free tier for live carbon prices
   - [EMBER](https://ember-climate.org/data/api/) - Free API for carbon intensity data
4. **Build valuation model**: Use stored data for Monte Carlo simulations